# Fast-fail analysis — AE trivial inversion (latent recovery)

**Question**: can the foundation model's latent space be navigated by gradient
optimization well enough to represent true velocity models? (the
reparameterized-FWI premise)

**Data**: `notebooks/data/` — extracted from MLflow experiment
`encdec.highres.inversion` by `notebooks/data/extract_from_mlflow.py`.
Provenance and column meanings: `notebooks/data/DATA_DICTIONARY.md`.

**Campaign**: 4 refit ML models (encdec 3×3 seed 7, encdec 5×5 seed 79,
encdec 7×7 seed 571, transencdec 5×5 seed 571) × 4 latent inits ×
5 lrs × 4 epoch budgets; every run inverts the SAME 3 target models —
the hardcoded inverted test indices [1, 10, 20], hand-picked difficult
examples from the refit's test set (whose 100-model MCCV holdout is
logged as context only). The extract carries the **323 ACTIVE runs** of
the experiment (standing rule: active only); **320 form the clean set**
(the complete factorial).

Reading order: audit the cleaning (cell 2) → sweep map (3) → final-state
tables (4) → per-model results (5) → baseline gap (6) → convergence &
best-so-far (7) → difficulty spread (8) → verdict (9).

In [5]:
from pathlib import Path
import pandas as pd

DATA = Path("data") if Path("data/runs.csv").exists() else Path("notebooks/data")
runs = pd.read_csv(DATA / "runs.csv", parse_dates=["start_time_iso"])

print("total runs:", len(runs), "(active only — standing run-lifecycle rule)")
display(runs["group"].value_counts().rename("runs").to_frame())

sweep = runs[runs["group"] == "sweep_full_samples"].copy()
kept = sweep[sweep["kept_in_clean_set"] == True].copy()     # noqa: E712
print(f"sweep {len(sweep)} = kept {len(kept)} (one run per combo)")
assert len(runs) == 323 and len(kept) == 320, "unexpected extract shape"
assert len(kept) == kept["combo"].nunique(), "kept must be one row per combo"
# Standing rule (user, 2026-11-24, valid forever): obsolete/play runs are
# deleted from MLflow before results are taken; kept test runs are renamed.
# The extract carries ACTIVE runs only — the 137 soft-deleted runs of this
# campaign are documented in DATA_DICTIONARY.md but are not in this data.

print("\nkept runs per refit ML model:")
display(kept.groupby(["arch_family", "kernel", "seed", "refit_short"])
        .size().rename("kept").to_frame())

total runs: 460


,runs
group,
sweep_full_samples,365
debug_50samples,68
param_less,24
tst_sanity,3


sweep 365 = kept 321 + duplicates 44

kept runs per refit model:


kept
arch_family kernel seed refit_short      
encdec      3      7    6d115baa       81
            5      79   9247757e       80
            7      571  b01fdd28       80
transencdec 5      571  32409900       80

In [6]:
# Sweep map: is the 4x5x4 factorial complete for every refit model?
INITS = ["trained", "zeros", "random", "randomn"]
LRS = [0.001, 0.01, 0.1, 0.5, 1.0]
BUDGETS = [50, 100, 500, 1000]
key_cols = ["refit_short", "latent_init", "lr", "max_epochs"]

present = set(map(tuple, kept[key_cols].itertuples(index=False, name=None)))
expected = {(r, i, float(lr), float(ep))
            for r in kept["refit_short"].unique()
            for i in INITS for lr in LRS for ep in BUDGETS}
print(f"grid cells present: {len(present & expected)} / {len(expected)}")
print("missing:", sorted(expected - present) or "none")

outside = kept[~kept[key_cols].apply(lambda r: tuple(r) in expected, axis=1)]
print("outside the 4x5x4 grid:", len(outside),
      "(none expected — the 200-ep probe was soft-deleted in the curation)")
assert len(outside) == 0, "unexpected non-grid runs in the active set"

# Duplicate re-runs are also absent from this extract by the standing rule:
# they were soft-deleted in the same manual curation. They were numerical
# replicates (same latent init; measured spread 0 except at the lr=1.0
# batch switch) — see DATA_DICTIONARY.md and docs/refactor-plan.md.

grid cells present: 320 / 320
missing: none

runs outside the 4x5x4 grid (exploratory):


,combo,inversion_mse_mean_test_set,start_time_iso
385,6d115baa|trained|lr=1.0|ep=200,0.00398,2026-11-12 18:16:46



clean set restricted to the grid: 321 -> 320 runs

the duplicate runs (dropped by keep-last) — note the random-init re-runs,
which are distinct latent draws (ADR-0002 J-draws material):


,combo,latent_init,lr,max_epochs,start_time_iso
349,9247757e|random|lr=0.5|ep=50,random,0.5,50.0,2026-11-12 20:10:47
350,b01fdd28|random|lr=0.5|ep=50,random,0.5,50.0,2026-11-12 20:10:47
348,32409900|random|lr=0.5|ep=50,random,0.5,50.0,2026-11-12 20:10:58
347,6d115baa|random|lr=0.5|ep=50,random,0.5,50.0,2026-11-12 20:10:59
381,6d115baa|random|lr=1.0|ep=50,random,1.0,50.0,2026-11-12 18:26:52
366,b01fdd28|random|lr=1.0|ep=50,random,1.0,50.0,2026-11-12 20:00:44
365,32409900|random|lr=1.0|ep=50,random,1.0,50.0,2026-11-12 20:01:10
364,9247757e|random|lr=1.0|ep=50,random,1.0,50.0,2026-11-12 20:02:50
363,6d115baa|random|lr=1.0|ep=50,random,1.0,50.0,2026-11-12 20:04:11
335,9247757e|random|lr=1.0|ep=50,random,1.0,50.0,2026-11-12 20:18:13


In [7]:
# Final-state tables. CAVEAT: 'final' = state at schedule end; with
# cosine_annealing_wr that can be a peak-lr point (quantified in the next cells).
mse = "inversion_mse_mean_test_set"
ssim = "inversion_ssim_mean_test_set"

for ep in [1000, 500, 100, 50]:
    sub = kept[kept["max_epochs"] == ep]
    t = sub.pivot_table(index="latent_init", columns="lr", values=mse, aggfunc="mean")
    print(f"\n=== final MSE @ {ep} epochs — mean over the 4 refit ML models "
          f"(each run = mean over its 3 inverted targets, the difficult "
          f"examples [1, 10, 20]) ===")
    display(t.round(4))

print("\n=== per-init best across all 20 (lr, budget) combos ===")
per_combo = kept.groupby(["latent_init", "lr", "max_epochs"])[mse].mean().reset_index()
display(per_combo.loc[per_combo.groupby("latent_init")[mse].idxmin()].round(4))

print("\n=== per refit ML model, clean grid (all budgets) ===")
display(kept.groupby(["arch_family", "kernel", "seed"])[mse]
        .agg(["count", "mean", "min", "max"]).round(4))

t = kept[kept["max_epochs"] == 1000].pivot_table(index="latent_init", columns="lr",
                                                 values=ssim, aggfunc="mean")
print("\n=== final SSIM @ 1000 epochs ===")
display(t.round(3))


=== final MSE @ 1000 epochs — mean over the 4 refit models (each run = mean over its 3 inverted targets, the difficult examples [1, 10, 20]) ===


lr,0.001,0.010,0.100,0.500,1.000
latent_init,,,,,
random,0.0056,0.0045,0.0045,0.0050,0.0053
randomn,0.0062,0.0046,0.0045,0.0047,0.0050
trained,0.0065,0.0044,0.0043,0.0044,0.0044
zeros,0.0047,0.0047,0.0060,0.0087,0.0105



=== final MSE @ 500 epochs — mean over the 4 refit models (each run = mean over its 3 inverted targets, the difficult examples [1, 10, 20]) ===


lr,0.001,0.010,0.100,0.500,1.000
latent_init,,,,,
random,0.0070,0.0048,0.0049,0.0057,0.0065
randomn,0.0077,0.0049,0.0047,0.0052,0.0055
trained,0.0088,0.0047,0.0044,0.0045,0.0046
zeros,0.0050,0.0051,0.0076,0.0120,0.0148



=== final MSE @ 100 epochs — mean over the 4 refit models (each run = mean over its 3 inverted targets, the difficult examples [1, 10, 20]) ===


lr,0.001,0.010,0.100,0.500,1.000
latent_init,,,,,
random,0.0134,0.0059,0.0063,0.0087,0.0104
randomn,0.0192,0.0067,0.0059,0.0073,0.0090
trained,0.0165,0.0068,0.0049,0.0053,0.0056
zeros,0.0068,0.0077,0.0137,0.0198,0.0225


... (4 outputs left out)


=== per refit model, clean grid (all budgets) ===


count    mean     min     max
arch_family kernel seed                               
encdec      3      7        80  0.0088  0.0037  0.0419
            5      79       80  0.0086  0.0044  0.0287
            7      571      80  0.0086  0.0047  0.0263
transencdec 5      571      80  0.0093  0.0044  0.0297


=== final SSIM @ 1000 epochs ===


lr,0.001,0.010,0.100,0.500,1.000
latent_init,,,,,
random,0.851,0.873,0.874,0.873,0.870
randomn,0.836,0.872,0.871,0.872,0.869
trained,0.824,0.869,0.871,0.872,0.873
zeros,0.874,0.870,0.858,0.831,0.820


In [8]:
# Per-example results: every run inverts the SAME 3 target examples (test-set positions
# [1, 10, 20]). The *_values MLflow metrics carry the per-example scalars
# (step t = t-th smallest inverted index — see log_evaluation_results);
# (vocabulary: 'model' = ML model; dataset items are 'examples' — legacy
#  identifiers keep the old wording: model_idx = example index)
# the extract maps them to model_idx in inversion_per_model.csv.
per_model = pd.read_csv(DATA / "inversion_per_model.csv")
pm = per_model[per_model["run_id"].isin(kept["run_id"])].copy()  # analysis set = kept (grid-only after cell 3)
print("per-example rows:", len(pm), "| runs:", pm["run_id"].nunique())

# (a) which target is hardest? per-example metrics averaged over all clean runs
print("=== per-example averages over all clean runs (difficulty of each target example) ===")
display(pm.groupby("model_idx")[["mse", "mae", "ssim"]].mean().round(4)
        .rename_axis(index="example_idx"))  # displayed as example_idx
# (b) per-model final MSE @ 1000 epochs: init x lr, one table per target
for idx, g in pm.groupby("model_idx"):
    sub = g[g["max_epochs"] == 1000]
    t = sub.pivot_table(index="latent_init", columns="lr", values="mse", aggfunc="mean")
    print(f"\n=== example (test-set position) {idx}: final MSE @ 1000 epochs ===")
    display(t.round(4))

# (c) the 3-example aggregate vs the per-example spread it hides
agg = kept.set_index("run_id")["inversion_mse_mean_test_set"].rename("agg_mean")
spread = pm.groupby("run_id")["mse"].agg(pm_min="min", pm_max="max")
c3 = pd.concat([agg, spread], axis=1).dropna()
c3["max/min"] = c3["pm_max"] / c3["pm_min"]
print("=== 3-example aggregate vs the per-example spread it hides ===")
display(c3.describe().round(3))

per-model rows: 960 | runs: 320
=== per-model averages over all clean runs (difficulty of each target) ===


,mse,mae,ssim
model_idx,,,
1,0.0060,0.0553,0.8485
10,0.0132,0.0948,0.8129
20,0.0073,0.0645,0.8390


KeyError: 'max_epochs'

In [ ]:
# Baseline gap: refit prediction (mean over its 100-example holdout) vs trivial
# inversion (mean over the 3 hand-picked DIFFICULT targets [1, 10, 20]).
# Difficulty mismatch inflates the gap; cell 7 (best-so-far) quantifies the
# final-state artifact on top of that.
base = pd.read_csv(DATA / "refit_baselines.csv").set_index("refit_short")

inv = kept[kept["max_epochs"] == 1000].pivot_table(
    index="refit_short", columns="latent_init", values=mse, aggfunc="mean")
cmp = pd.DataFrame({
    "refit_mse": base["mse_mean_test_set"],
    "inv@1000_mean_over_inits": inv.mean(axis=1),
    "inv@1000_best_init": inv.min(axis=1),
})
cmp["ratio_mean"] = cmp["inv@1000_mean_over_inits"] / cmp["refit_mse"]
cmp["refit_ssim"] = base["ssim_mean_test_set"]
inv_ssim = kept[kept["max_epochs"] == 1000].pivot_table(
    index="refit_short", columns="latent_init", values=ssim, aggfunc="mean")
cmp["inv_ssim@1000_mean"] = inv_ssim.mean(axis=1)

print("=== refit prediction vs trivial inversion ===")
display(cmp.round(4))

In [ ]:
# Convergence + best-so-far vs final (quantifies the warm-restart artifact).
import matplotlib.pyplot as plt

hist = pd.read_csv(DATA / "train_loss_history.csv")
print("history rows:", len(hist))

# clean curve: last logged value per step (steps may repeat)
curves = (hist.sort_values("step")
              .groupby(["run_id", "step"], as_index=False)["value"].last())
curves = curves.merge(kept[["run_id", "combo", "refit_short", "latent_init",
                            "lr", "max_epochs"]], on="run_id")

KEY = [("trained", 0.01), ("trained", 0.1), ("random", 0.01), ("random", 0.1)]
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, (init, lr) in zip(axes.ravel(), KEY):
    sel = curves[(curves["latent_init"] == init) & (curves["lr"] == lr)
                 & (curves["max_epochs"] == 1000)]
    for refit, g in sel.groupby("refit_short"):
        ax.plot(g["step"], g["value"], lw=0.8, label=refit)
    ax.set_yscale("log")
    ax.set_title(f"init={init}, lr={lr}, 1000 epochs")
    ax.set_xlabel("step")
    ax.set_ylabel("train_loss")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

bsf = curves.groupby("run_id")["value"].min().rename("best_so_far")
bf = pd.concat([bsf, kept.set_index("run_id")[mse].rename("final_mse")],
               axis=1).dropna()
bf["final/best"] = bf["final_mse"] / bf["best_so_far"]
print("=== final-state penalty over all 320 clean runs ===")
display(bf.describe().round(4))
print("\nworst final/best cases:")
display(bf.sort_values("final/best", ascending=False).head(10).round(4))
print("\nbest-so-far floor vs the refit ML models' prediction MSE (~0.0009):", bf["best_so_far"].min())

In [ ]:
# Per-example difficulty spread (comment i): MSE varies a lot across test examples,
# so single-example readings must be difficulty-matched. Here the spread is over
# the 3 inverted targets ([1, 10, 20]) — a small sample, but it brackets the
# difficulty range those specific models span.
spread = kept[["combo", "latent_init", "lr", "max_epochs",
               "inversion_mse_min_test_set", mse,
               "inversion_mse_max_test_set", "inversion_mse_std_test_set"]].copy()
spread["max/min"] = spread["inversion_mse_max_test_set"] / spread["inversion_mse_min_test_set"]

print("per-run spread of per-example MSE over the 3 inverted targets ([1, 10, 20]):")
display(spread[["max/min", "inversion_mse_std_test_set"]].describe().round(3))

refit_spread = (base["mse_max_test_set"] / base["mse_min_test_set"]).rename("refit max/min")
print("\nrefit baseline spread on the same holdouts:")
display(refit_spread.round(2).to_frame())

ax = spread["max/min"].plot(kind="hist", bins=40, figsize=(7, 3),
                            title="per-run max/min of per-model MSE")
ax.set_xlabel("max/min")
plt.show()

## Verdict — *draft after exploring*

**Findings**
- (fill after running the cells — the numbers you trust are the ones you produced)

**Caveats**
- final-state reporting at warm-restart peaks → quantified in the convergence cell; fix lands in the final experiments
- per-model difficulty variance → spread cell; single-example readings must be difficulty-matched

**Next (post-refactor AE+VAE rerun checklist)**
- best-so-far / cold-end reporting
- `init=encoder-of-target` — measures the manifold-coverage ceiling
- per-example paired analysis
- 10-seed MCCV (ADR-0002)